# B2-022-probabilistic-latent-models — Practice p09 — Solution

**Type:** constrained-coding · **Difficulty:** core · **Concepts:** kl-divergence

*50 minutes.*  
**Set:** B  
**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20260927`  
**Qualified prerequisites:** `aio-ml:F1-scientific-python`, `aio-ml:F4-matrices`, `aio-ml:F3-multivar-calculus`, `aio-ml:F5-probability`, `aio-ml:F6-svd-spectral`, `aio-ml:C1-ml-fundamentals`, `aio-ml:C2-linear-models`, `aio-ml:C6-neural-networks`, `aio-ml:C7-pytorch`, `aio-ml:C9-dimensionality-reduction`, `aio-ml:C8-neural-training`, `B2-021-cross-modal-transformers-vision`  
**Remediation links actually used:** [aio-ml:F1-scientific-python](../../../../aio-ml/units/01-F1-scientific-python/lesson.ipynb), [aio-ml:F4-matrices](../../../../aio-ml/units/04-F4-matrices/lesson.ipynb), [aio-ml:F3-multivar-calculus](../../../../aio-ml/units/03-F3-multivar-calculus/lesson.ipynb), [aio-ml:F5-probability](../../../../aio-ml/units/05-F5-probability/lesson.ipynb), [aio-ml:F6-svd-spectral](../../../../aio-ml/units/12-F6-svd-spectral/lesson.ipynb), [aio-ml:C1-ml-fundamentals](../../../../aio-ml/units/06-C1-ml-fundamentals/lesson.ipynb), [aio-ml:C2-linear-models](../../../../aio-ml/units/07-C2-linear-models/lesson.ipynb), [aio-ml:C6-neural-networks](../../../../aio-ml/units/11-C6-neural-networks/lesson.ipynb), [aio-ml:C7-pytorch](../../../../aio-ml/units/13-C7-pytorch/lesson.ipynb), [aio-ml:C9-dimensionality-reduction](../../../../aio-ml/units/16-C9-dimensionality-reduction/lesson.ipynb), [aio-ml:C8-neural-training](../../../../aio-ml/units/15-C8-neural-training/lesson.ipynb), [B2-021-cross-modal-transformers-vision](../../B2-021-cross-modal-transformers-vision/lesson.ipynb).

## Supplied cell (copied verbatim from the statement)

In [ ]:
import math
import torch

SEED = 20260927
mu = torch.tensor([[0.0, 0.0],
                   [1.0, -0.5],
                   [0.3, 2.0]], dtype=torch.float64)
logvar = torch.tensor([[0.0, 0.0],
                       [math.log(0.25), math.log(2.0)],
                       [-1.0, 0.5]], dtype=torch.float64)
expected = torch.tensor([0.0, 1.0965735903, 2.3033003559], dtype=torch.float64)
N_MC = 200_000
MC_ATOL = 0.03

## Solution

Closed form, per example: $\mathrm{KL}=\tfrac12\sum_j(\mu_j^2+e^{\ell_j}-\ell_j-1)$.

Monte Carlo: $\mathrm{KL}(q\|p)=E_{z\sim q}[\log q(z)-\log p(z)]$, with the diagonal log-density
$\log N(z;m,\operatorname{diag}e^{\ell})=\sum_j-\tfrac12\big(\log2\pi+\ell_j+(z_j-m_j)^2e^{-\ell_j}\big)$ and $p=N(0,I)$ ($m=0$, $\ell=0$).
One generator is shared across rows 0, 1, 2 in order, as required.

In [ ]:
def kl_to_standard_normal(mu, logvar):
    if mu.dim() != 2 or logvar.dim() != 2:
        raise ValueError("mu and logvar must be 2-D (B, k)")
    if mu.shape != logvar.shape:
        raise ValueError("mu and logvar shapes differ")
    return 0.5 * (mu ** 2 + logvar.exp() - logvar - 1).sum(dim=1)


def diag_gaussian_log_density(z, mean, logvar):
    return (-0.5 * (math.log(2 * math.pi) + logvar + (z - mean) ** 2 / logvar.exp())).sum(dim=-1)


def monte_carlo_kl(mu_row, logvar_row, n, generator):
    k = mu_row.shape[0]
    eps = torch.randn(n, k, generator=generator, dtype=torch.float64)
    z = mu_row + torch.exp(0.5 * logvar_row) * eps
    log_q = diag_gaussian_log_density(z, mu_row, logvar_row)
    log_p = diag_gaussian_log_density(z, torch.zeros_like(mu_row), torch.zeros_like(logvar_row))
    return (log_q - log_p).mean().item()


closed = kl_to_standard_normal(mu, logvar)
generator = torch.Generator().manual_seed(SEED)
mc = torch.tensor([monte_carlo_kl(mu[i], logvar[i], N_MC, generator) for i in range(3)], dtype=torch.float64)

# Wrong variants (local only)
def kl_variant_no_minus_one(mu, logvar):
    return 0.5 * (mu ** 2 + logvar.exp() - logvar).sum(dim=1)

def kl_variant_std_for_var(mu, logvar):
    return 0.5 * (mu ** 2 + torch.exp(0.5 * logvar) - logvar - 1).sum(dim=1)

def passes_probe2(values):
    return bool(((values - mc).abs() <= MC_ATOL).all())

variant_a = kl_variant_no_minus_one(mu, logvar)
variant_b = kl_variant_std_for_var(mu, logvar)
print(closed, mc, variant_a, variant_b)

**Probe 4.** Each summand $\tfrac12(e^{\ell_j}-\ell_j-1)\ge0$ by the Session 2 lemma $\log x\le x-1$ applied to $x=e^{\ell_j}$ (i.e. $\ell_j\le e^{\ell_j}-1$), and $\tfrac12\mu_j^2\ge0$, so every closed-form value is nonnegative term by term.

### Answer check

In [ ]:
# Checks: the KL-divergence answers against pinned values
assert closed.shape == (3,) and closed.dtype == torch.float64
assert torch.allclose(closed, expected, atol=1e-9, rtol=1e-9)
assert passes_probe2(closed)
assert not passes_probe2(variant_a)
assert not passes_probe2(variant_b)
assert bool((closed >= 0).all())
assert bool((0.5 * (logvar.exp() - logvar - 1) >= 0).all())
try:
    kl_to_standard_normal(mu, logvar[:, :1])
    raise AssertionError("shape mismatch accepted")
except ValueError:
    pass